In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.tree import DecisionTreeClassifier

In [38]:
df = pd.read_csv('/kaggle/input/datasets/sanjanbm/titanic-train-dataset/train.csv')

In [39]:
df.sample(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
765,766,1,1,"Hogeboom, Mrs. John C (Anna Andrews)",female,51.0,1,0,13502,77.9583,D11,S
130,131,0,3,"Drazenoic, Mr. Jozef",male,33.0,0,0,349241,7.8958,NaN,C
252,253,0,1,"Stead, Mr. William Thomas",male,62.0,0,0,113514,26.5500,C87,S
844,845,0,3,"Culumovic, Mr. Jeso",male,17.0,0,0,315090,8.6625,NaN,S


In [40]:
df.drop(columns = ['PassengerId','Name','Ticket','Cabin'], inplace = True)
df.sample(4)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
696,0,3,male,44.0,0,0,8.0500,S
435,1,1,female,14.0,1,2,120.0000,S
63,0,3,male,4.0,3,2,27.9000,S
519,0,3,male,32.0,0,0,7.8958,S


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [42]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size = 0.2, random_state=42)

X_train.shape, X_test.shape

((712, 7), (179, 7))

In [44]:
si = SimpleImputer()
si_embarked = SimpleImputer(strategy='most_frequent')

X_train_age = pd.DataFrame(si.fit_transform(X_train[['Age']]), 
                           columns=['Age'], 
                           index=X_train.index
                          )

X_train_embarked = pd.DataFrame(si_embarked.fit_transform(X_train[['Embarked']]), 
                                columns=['Embarked'],
                               index=X_train.index
                                )

X_test_age = pd.DataFrame(si.transform(X_test[['Age']]),
                         columns=['Age'],
                         index=X_test.index
                         )

X_test_embarked = pd.DataFrame(si_embarked.transform(X_test[['Embarked']]), 
                              columns=['Embarked'],
                              index=X_test.index
                              )

X_train_age.shape, X_test_age.shape

((712, 1), (179, 1))

In [45]:
X_train_embarked.head()

,Embarked
331,S
733,S
382,S
704,S
813,S


In [46]:
X_test_age.head()

,Age
709,29.498846
439,31.000000
840,20.000000
720,6.000000
39,14.000000


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [50]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn import set_config

# 1. This tells ALL scikit-learn transformers to return DataFrames
set_config(transform_output="pandas")

# 2. Initialize the Encoders
# sparse_output=False ensures it doesn't create those <Compressed...> objects
ohe_sex = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_embarked = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 3. Fit and Transform Training Data
# No need for pd.DataFrame() wrapper anymore!
X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train[['Embarked']])

# 4. Transform Test Data
X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.transform(X_test[['Embarked']])

# Check the results
print(X_train_sex.head())
print(X_train_embarked.head())

     Sex_female  Sex_male
331         0.0       1.0
733         0.0       1.0
382         0.0       1.0
704         0.0       1.0
813         1.0       0.0
     Embarked_C  Embarked_Q  Embarked_S  Embarked_nan
331         0.0         0.0         1.0           0.0
733         0.0         0.0         1.0           0.0
382         0.0         0.0         1.0           0.0
704         0.0         0.0         1.0           0.0
813         0.0         0.0         1.0           0.0


In [53]:
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'])

print(X_train_rem.head())
print(X_test_rem.head())

     Pclass  SibSp  Parch     Fare
331       1      0      0  28.5000
733       2      0      0  13.0000
382       3      0      0   7.9250
704       3      1      0   7.8542
813       3      4      2  31.2750
     Pclass  SibSp  Parch     Fare
709       3      1      1  15.2458
439       2      0      0  10.5000
840       3      0      0   7.9250
720       2      0      1  33.0000
39        3      1      0  11.2417


In [60]:
X_train_transformed = pd.concat([X_train_rem, X_train_sex, X_train_age, X_train_embarked], axis = 1)
X_test_transformed = pd.concat([X_test_rem, X_test_sex, X_test_age, X_test_embarked], axis = 1)

print(X_train_transformed.head())
print(X_test_transformed.head())

X_train_transformed = X_train_transformed.drop(columns=['Embarked_nan']) 
X_test_transformed = X_test_transformed.drop(columns=['Embarked_nan']) 

     Pclass  SibSp  Parch     Fare  Sex_female  Sex_male   Age  Embarked_C  \
331       1      0      0  28.5000         0.0       1.0  45.5         0.0   
733       2      0      0  13.0000         0.0       1.0  23.0         0.0   
382       3      0      0   7.9250         0.0       1.0  32.0         0.0   
704       3      1      0   7.8542         0.0       1.0  26.0         0.0   
813       3      4      2  31.2750         1.0       0.0   6.0         0.0   

     Embarked_Q  Embarked_S  Embarked_nan  
331         0.0         1.0           0.0  
733         0.0         1.0           0.0  
382         0.0         1.0           0.0  
704         0.0         1.0           0.0  
813         0.0         1.0           0.0  
     Pclass  SibSp  Parch     Fare  Sex_female  Sex_male        Age  \
709       3      1      1  15.2458         0.0       1.0  29.498846   
439       2      0      0  10.5000         0.0       1.0  31.000000   
840       3      0      0   7.9250         0.0       1

In [61]:
X_train_transformed.shape

(712, 10)

In [64]:
y_train.sample()
print(y_train.shape)

(712,)


In [66]:
clf = DecisionTreeClassifier()

clf.fit(X_train_transformed, y_train)


DecisionTreeClassifier()

In [68]:
from sklearn.metrics import accuracy_score

y_pred = clf.predict(X_test_transformed)

print(accuracy_score(y_test, y_pred)*100)

78.2122905027933


In [70]:
import pickle

pickle.dump(ohe_sex, open('ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked, open('ohe_embarked.pkl','wb'))
pickle.dump(clf, open('clf.pkl','wb'))